# Producción Agrícola (Procesamiento)

En esta libreta prepararemos los datos para el futuro analisis exploratorio de datos.

In [1]:
from scripts.config import PRELOADS
import pickle

# Notemos que ahora estamos obteniendo la información ya normalizada.
with open(PRELOADS['AGROPROD_1'], "rb") as preprocess: 
    dgsiap = pickle.load(preprocess)

dgsiap

En esta etapa (/interm a /process) nos permitimos modificar el diccionario y los datos para mejorar su uso para el analisis de datos.

## Diccionario de Datos.

Partiremos con el Diccionario de Datos. Modificando las reglas definidas por la fuente oficial. Para ello ocuparemos el catalogo y los tipos de datos definidas.

In [21]:
codes = dgsiap.root
data_vars = dgsiap.subroot

catalog = codes.open(pars=[1, {"index_col":0}])
print(data_vars)
catalog

{'T': 'Texto', 'N': 'Numérico entero', 'D': 'Numérico decimal', '#': 'Carácter numérico.', 'A': 'Carácter alfanumérico.'}


,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Anio,N,4,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
Idestado,N,2,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
Nomestado,T,60,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
Idddr,N,3,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
Nomddr,T,60,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
Idcader,N,2,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
Nomcader,T,90,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
Idmunicipio,T,3,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
Nommunicipio,T,45,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...


### Correccion de Tipos
Para nuestro diccionario usaremos en gran parte los tipos de datos dados por la libreria pandas. Para ello usaremos el primer conjunto de datos como referencia y forzaremos el tipo en el resto.


In [ ]:
test = dgsiap.branch[0]
df = test.open(pars=[1, {"index_col":0, "low_memory":False}])
df.dtypes

       Anio  Idestado       Nomestado  Idddr          Nomddr  Idcader  \
0      2025         1  Aguascalientes      1  Aguascalientes        1   
1      2025         1  Aguascalientes      1  Aguascalientes        1   
2      2025         1  Aguascalientes      1  Aguascalientes        1   
3      2025         1  Aguascalientes      1  Aguascalientes        1   
4      2025         1  Aguascalientes      1  Aguascalientes        1   
...     ...       ...             ...    ...             ...      ...   
35897  2025        32       Zacatecas    190           Jalpa        4   
35898  2025        32       Zacatecas    190           Jalpa        4   
35899  2025        32       Zacatecas    190           Jalpa        4   
35900  2025        32       Zacatecas    190           Jalpa        4   
35901  2025        32       Zacatecas    190           Jalpa        4   

             Nomcader  Idmunicipio    Nommunicipio  Idciclo  ... Nomunidad  \
0      Aguascalientes            1  Aguascali

Primero definiremos el tipo de dato de pandas que usará cada variable a partir de ahora.

In [ ]:
import pandas as pd

agro_varsguide = pd.DataFrame(index = data_vars.keys(), columns = ['Tipo Pandas', 'Descripción'])

for col in df.columns:
    # print(col)
    var_type = catalog.loc[col, 'Tipo']
    agro_varsguide.loc[var_type] = [df.dtypes[col], data_vars[var_type]]

agro_varsguide=agro_varsguide.dropna(axis=0)
agro_varsguide

,Tipo Pandas,Descripción
T,str,Texto
N,int64,Numérico entero
D,float64,Numérico decimal


Ahora haremos la conexión con las columnas.

In [ ]:
# Comenzamos con la conexión general.
agro_dtypes = []
for col in df.columns:
    var_type = catalog.loc[col, 'Tipo']
    agro_dtypes.append(agro_varsguide.loc[var_type, 'Tipo Pandas'])


data_checker = pd.Series(index = df.columns, data=agro_dtypes)
data_checker

Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio               str
Nommunicipio              str
Idciclo                 int64
Nomcicloproductivo        str
Idmodalidad             int64
Nommodalidad              str
Idunidadmedida          int64
Nomunidad                 str
Idcultivo               int64
Nomcultivo                str
Sembrada              float64
Cosechada             float64
Siniestrada           float64
Volumenproduccion     float64
Rendimiento           float64
Precio                float64
Valorproduccion       float64
dtype: object

# Casos especiales.

Como se comentó en la etapa anterior. Idmunicipio no tiene argumentos para ser de tipo texto cuando el resto de identificadores son tipo entero. Entonces haremos el cambio.

In [33]:
data_checker['Idmunicipio'] = agro_varsguide.loc['N', 'Tipo Pandas']
data_checker

Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio             int64
Nommunicipio              str
Idciclo                 int64
Nomcicloproductivo        str
Idmodalidad             int64
Nommodalidad              str
Idunidadmedida          int64
Nomunidad                 str
Idcultivo               int64
Nomcultivo                str
Sembrada              float64
Cosechada             float64
Siniestrada           float64
Volumenproduccion     float64
Rendimiento           float64
Precio                float64
Valorproduccion       float64
dtype: object

Ya tenemos los tipos definidos. Ahora falta corroborar que estos coinciden con el resto.

In [39]:
for test in dgsiap.branch:
    df = test.open(pars=[1, {"index_col":0, "low_memory":False}])
    u = df.dtypes
    try:
        if (u!= data_checker).any():
            continue
    except:
        print("A",u)
        print("B",data_checker)

A Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio             int64
Nommunicipio              str
Idciclo                 int64
Nomcicloproductivo        str
Idmodalidad             int64
Nommodalidad              str
Idunidadmedida          int64
Nomunidad                 str
Idcultivo               int64
Nomcultivo Sin Um         str
Sembrada              float64
Cosechada             float64
Siniestrada           float64
Volumenproduccion     float64
Rendimiento           float64
Precio                float64
Valorproduccion       float64
dtype: object
B Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio             int64
Nommunicipio          